In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
# Install required packages
!pip install -q dagshub mlflow

import dagshub
import mlflow
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
DAGSHUB_TOKEN = user_secrets.get_secret("GUGA_TOKEN")
DAGSHUB_USERNAME = user_secrets.get_secret("GUGA_USERNAME")

repo_name = "ML_assignment1" 

mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USERNAME}/{repo_name}.mlflow")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

print(" Connected to DagsHub + MLflow successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 5.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 65.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [3]:
import pandas as pd

train_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

mlflow.set_experiment("House_Price_Analysis")

print(f"Data Loaded! Train shape: {train_df.shape}")

Data Loaded! Train shape: (1460, 81)


# 1. Data Cleaning & Missing Value Handling

In [4]:
import numpy as np

cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
train_df = train_df.drop(columns=[col for col in cols_to_drop if col in train_df.columns])

X = train_df.drop(['SalePrice', 'Id'], axis=1)
y = train_df['SalePrice']

print(" Data split and ready for encoding.")

 Data split and ready for encoding.


# 2. Feature Engineering: Categorical Encoding

In [5]:
X_cleaned = X.copy()

num_cols = X_cleaned.select_dtypes(include=[np.number]).columns
X_cleaned[num_cols] = X_cleaned[num_cols].fillna(X_cleaned[num_cols].median())

cat_cols = X_cleaned.select_dtypes(include=['object']).columns

for col in cat_cols:
    X_cleaned[col] = X_cleaned[col].fillna("None")
    
    X_cleaned[col] = pd.factorize(X_cleaned[col])[0]

print(f"Shape: {X_cleaned.shape}")  
missing_total = X_cleaned.isnull().sum().sum()
print(f"Total missing values in X_cleaned: {missing_total}")
print(" Data is now 100% numeric and RFE-ready.")

Shape: (1460, 75)
Total missing values in X_cleaned: 0
 Data is now 100% numeric and RFE-ready.


# 3. Feature Selection: Recursive Feature Elimination (RFE)

In [6]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_cleaned, y, test_size=0.2, random_state=42)

selector = RFE(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    n_features_to_select=30
)

selector.fit(X_train, y_train)

selected_columns = X_train.columns[selector.support_].tolist() 


X_train_sel = X_train[selected_columns]
X_val_sel = X_val[selected_columns]

print(f" Done! Picked {len(selected_columns)} columns.")
print(f"Top 5: {selected_columns[:5]}")

 Done! Picked 30 columns.
Top 5: ['MSSubClass', 'LotFrontage', 'LotArea', 'Neighborhood', 'OverallQual']


# 4. Model Training & MLflow Experiments

## 4.1 Baseline Linear Regression

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

with mlflow.start_run(run_name="Baseline_Linear"):
    model_lin = LinearRegression()
    model_lin.fit(X_train, y_train)

    train_preds = np.abs(model_lin.predict(X_train))
    train_rmse = np.sqrt(mean_squared_error(np.log(y_train), np.log(train_preds)))

    val_preds = np.abs(model_lin.predict(X_val))
    val_rmse = np.sqrt(mean_squared_error(np.log(y_val), np.log(val_preds)))
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("GAP", val_rmse - train_rmse)
    
    print(f"--- Experiment 1: Baseline Linear ---")
    print(f"Train RMSE: {train_rmse:.5f}")
    print(f"Validation RMSE: {val_rmse:.5f}")
    print(f"The GAP is: {val_rmse - train_rmse:.5f}")

--- Experiment 1: Baseline Linear ---
Train RMSE: 0.16353
Validation RMSE: 0.20709
The GAP is: 0.04356
🏃 View run Baseline_Linear at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0/runs/93dd6dbca318486db476a571ac978e91
🧪 View experiment at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0


## 4.2 Single Decision Tree

In [8]:
from sklearn.tree import DecisionTreeRegressor
with mlflow.start_run(run_name="Single_Decision_Tree"):
    dtree = DecisionTreeRegressor(max_depth=10, random_state=42)
    dtree.fit(X_train, y_train)
    
    t_preds = dtree.predict(X_train)
    t_rmse = np.sqrt(mean_squared_error(np.log(y_train), np.log(t_preds)))
    
    v_preds = dtree.predict(X_val)
    v_rmse = np.sqrt(mean_squared_error(np.log(y_val), np.log(val_preds)))
    
    mlflow.log_param("model_type", "SingleDecisionTree")
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("train_rmse", t_rmse)
    mlflow.log_metric("val_rmse", v_rmse)
    mlflow.log_metric("GAP", v_rmse - t_rmse)
    
    print(f"--- Experiment: Single Decision Tree ---")
    print(f"Train RMSE: {t_rmse:.5f}")
    print(f"Validation RMSE: {v_rmse:.5f}")
    print(f"The GAP: {v_rmse - t_rmse:.5f}")

--- Experiment: Single Decision Tree ---
Train RMSE: 0.05880
Validation RMSE: 0.20709
The GAP: 0.14830
🏃 View run Single_Decision_Tree at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0/runs/8f40de1827b04f63903acc9e331042bc
🧪 View experiment at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0


## 4.3  Lasso Regression

In [9]:
from sklearn.linear_model import Lasso

with mlflow.start_run(run_name="Experiment_Lasso"):
    model_lasso = Lasso(alpha=0.1, random_state=42)
    model_lasso.fit(X_train_sel, y_train)
    
    train_preds = np.abs(model_lasso.predict(X_train_sel))
    train_rmse = np.sqrt(mean_squared_error(np.log(y_train), np.log(train_preds)))
    
    val_preds = np.abs(model_lasso.predict(X_val_sel))
    val_rmse = np.sqrt(mean_squared_error(np.log(y_val), np.log(val_preds)))

    mlflow.log_param("model_type", "LassoRegression")
    mlflow.log_param("alpha", 0.1)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("GAP", val_rmse - train_rmse)
    
    print(f"--- Experiment: Lasso Regression ---")
    print(f"Train RMSE: {train_rmse:.5f}")
    print(f"Validation RMSE: {val_rmse:.5f}")
    print(f"The GAP: {val_rmse - train_rmse:.5f}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.408e+11, tolerance: 6.967e+08
  model = cd_fast.enet_coordinate_descent(


--- Experiment: Lasso Regression ---
Train RMSE: 0.18960
Validation RMSE: 0.25079
The GAP: 0.06119
🏃 View run Experiment_Lasso at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0/runs/626a71b4af314c4b83a10e2212b215b5
🧪 View experiment at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0


## 4.5  Random Forest 


In [10]:
with mlflow.start_run(run_name="Tuned_RF"):
    # Using the same 30 features
    X_train_sel = X_train[selected_columns]
    X_val_sel = X_val[selected_columns]
    
    rf_params = {
        "n_estimators": 500,   
        "max_depth": 15,       
        "min_samples_split": 5, 
        "random_state": 42
    }
    
    model_rf_final = RandomForestRegressor(**rf_params)
    model_rf_final.fit(X_train_sel, y_train)
    
    train_preds_final = model_rf_final.predict(X_train_sel)
    train_rmse_final = np.sqrt(mean_squared_error(np.log(y_train), np.log(train_preds_final)))
    
    val_preds_final = model_rf_final.predict(X_val_sel)
    val_rmse_final = np.sqrt(mean_squared_error(np.log(y_val), np.log(val_preds_final)))

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_params(rf_params)
    mlflow.log_metric("train_rmse", train_rmse_final)
    mlflow.log_metric("val_rmse", val_rmse_final)
    mlflow.log_metric("GAP", val_rmse_final - train_rmse_final)
    
    print(f"--- Experiment: RFE Tuned Model ---")
    print(f"Train RMSE: {train_rmse_final:.5f}")
    print(f"Validation RMSE: {val_rmse_final:.5f}")
    print(f"The GAP: {val_rmse_final - train_rmse_final:.5f}")

--- Experiment: RFE Tuned Model ---
Train RMSE: 0.06676
Validation RMSE: 0.15229
The GAP: 0.08553
🏃 View run Tuned_RF at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0/runs/53c3bfd4c08d4a83b7065dd0fa5821cf
🧪 View experiment at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0


## 4.4 XGBRegressor


In [11]:
from xgboost import XGBRegressor

with mlflow.start_run(run_name="XGBoost_Final"):
    xgb_params = {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 4,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42
    }
    
    model_xgb = XGBRegressor(**xgb_params)
    model_xgb.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  verbose=False)
    
    train_preds_xgb = model_xgb.predict(X_train)
    train_rmse_xgb = np.sqrt(mean_squared_error(np.log(y_train), np.log(train_preds_xgb)))
    
    val_preds_xgb = model_xgb.predict(X_val)
    val_rmse_xgb = np.sqrt(mean_squared_error(np.log(y_val), np.log(val_preds_xgb)))
    
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_params(xgb_params)
    mlflow.log_metric("train_rmse", train_rmse_xgb)
    mlflow.log_metric("val_rmse", val_rmse_xgb)
    mlflow.log_metric("GAP", val_rmse_xgb - train_rmse_xgb)
    
    print(f"--- Experiment: XGBoost ---")
    print(f"Train RMSE: {train_rmse_xgb:.5f}")
    print(f"Validation RMSE: {val_rmse_xgb:.5f}")
    print(f"The GAP: {val_rmse_xgb - train_rmse_xgb:.5f}")

--- Experiment: XGBoost ---
Train RMSE: 0.03747
Validation RMSE: 0.13466
The GAP: 0.09720
🏃 View run XGBoost_Final at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0/runs/91a2b3ae3f24481698841223fe5d9f36
🧪 View experiment at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0


# 5. Model Registration

In [13]:
import mlflow.sklearn

# We register the 'model_rf_final' because it had the best balance (0.15413)
model_name = "HousePriceBestModel"

with mlflow.start_run(run_name="Registration_Run"):
    mlflow.sklearn.log_model(
        sk_model=model_xgb,         
        name="model",
        registered_model_name=model_name
    )
    mlflow.log_params(xgb_params) 
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_metric("train_rmse", train_rmse_xgb) 
    mlflow.log_metric("val_rmse", val_rmse_xgb)      
    print(f" Success! Version of '{model_name}' has been registered.")

2026/04/13 09:09:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'HousePriceBestModel' already exists. Creating a new version of this model...
2026/04/13 09:09:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: HousePriceBestModel, version 58
Created version '58' of model 'HousePriceBestModel'.


 Success! Version of 'HousePriceBestModel' has been registered.
🏃 View run Registration_Run at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0/runs/0738df615604411b8022fd7142cb3430
🧪 View experiment at: https://dagshub.com/ggzob23/ML_assignment1.mlflow/#/experiments/0
